# Improved Pipeline – DIS22 ArXiv Warriors
**Kaskade: Regex → spaCy → Ollama-LLM**

Forschungsfrage: In welchem Ausmaß überschneiden sich die Informationen in Tabellen mit dem umliegenden Fließtext in bioRxiv-Preprints?

Labels: `info_in_text` | `partial` | `only_table` | `no_refs`

## 0. Setup & Abhängigkeiten installieren

In [1]:
# Einmalig ausführen – installiert fehlende Pakete in die AKTUELLE Python-Umgebung
import sys
!{sys.executable} -m pip install spacy scikit-learn pandas tqdm ollama -q
!{sys.executable} -m spacy download en_core_web_sm -q
print('✓ Pakete installiert')


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
✓ Pakete installiert


## 1. Imports & Pfade

In [2]:
import json
import re
import pathlib
import pandas as pd
from tqdm.auto import tqdm

# Pfade – automatisch relativ zum Notebook
notebook_path = globals().get('__vsc_ipynb_file__')
NOTEBOOK_DIR = pathlib.Path(notebook_path).resolve().parent if notebook_path else pathlib.Path.cwd()
PROJECT_DIR  = NOTEBOOK_DIR.parent
OUTPUT_DIR   = PROJECT_DIR / 'SampleData' / 'output'
ANNOT_PATH   = PROJECT_DIR / 'annotation' / 'alle_annotiert.csv'
RESULTS_PATH = PROJECT_DIR / 'results_improved.csv'

json_files = sorted(OUTPUT_DIR.glob('*.json'))
print(f'Notebook-Verzeichnis : {NOTEBOOK_DIR}')
print(f'Output-Verzeichnis   : {OUTPUT_DIR}')
print(f'JSON-Dateien gefunden: {len(json_files)}')
print(f'Annotation vorhanden : {ANNOT_PATH.exists()}')

Notebook-Verzeichnis : /Users/cemilhantozak/Documents/Studium/6. Semester/DIS22 - Projekt ||/Code
Output-Verzeichnis   : /Users/cemilhantozak/Documents/Studium/6. Semester/DIS22 - Projekt ||/SampleData/output
JSON-Dateien gefunden: 499
Annotation vorhanden : False


## 2. spaCy & Ollama laden

In [4]:
# --- spaCy (Stufe 2) ---
try:
    import spacy
    try:
        _nlp = spacy.load('en_core_web_md')
        print('✓ spaCy-Modell: en_core_web_md')
    except OSError:
        _nlp = spacy.load('en_core_web_sm')
        print('✓ spaCy-Modell: en_core_web_sm')
    SPACY_OK = True
except Exception as e:
    print(f'✗ spaCy nicht verfügbar: {e}')
    _nlp = None
    SPACY_OK = False

# --- Ollama (Stufe 3) ---
_PREFERRED_MODELS = [
    'llama3.3', 'qwen2.5:72b', 'deepseek-r1:32b',
    'llama3.1:70b', 'llama3.1:8b', 'llama3.2', 'mistral',
]
_ollama_model = None
OLLAMA_OK = False

try:
    import ollama as _ollama_lib
    _model_list = _ollama_lib.list()
    _available = (
        [m.model for m in _model_list.models]
        if hasattr(_model_list, 'models')
        else [m['name'] for m in _model_list.get('models', [])]
    )
    if _available:
        for pref in _PREFERRED_MODELS:
            match = next((a for a in _available if a.startswith(pref)), None)
            if match:
                _ollama_model = match
                break
        if not _ollama_model:
            _ollama_model = _available[0]
        OLLAMA_OK = True
        print(f'✓ Ollama verfügbar. Modell: {_ollama_model}')
        print(f'  Alle Modelle: {_available}')
    else:
        print('✗ Ollama läuft, aber keine Modelle installiert.')
        print('  → Terminal: ollama pull llama3.1:8b')
except ConnectionError:
    print('✗ Ollama: Server nicht erreichbar.')
    print('  → Terminal: ollama serve   (offen lassen, dann Zelle nochmal ausführen)')
except Exception as e:
    print(f'✗ Ollama nicht verfügbar: {type(e).__name__}: {e}')

print(f'\nKaskade: Regex', end='')
print(f' → spaCy' if SPACY_OK else ' (kein spaCy)', end='')
print(f' → LLM ({_ollama_model})' if OLLAMA_OK else ' (kein LLM)', end='')
print()

✓ spaCy-Modell: en_core_web_sm
✓ Ollama verfügbar. Modell: llama3.1:8b
  Alle Modelle: ['llama3.1:8b']

Kaskade: Regex → spaCy → LLM (llama3.1:8b)


## 3. Hilfsfunktionen

In [5]:
# --- Tabellen-Parsing ---

def extract_table_content(table: dict) -> tuple:
    """Gibt (all_cells_text, is_complete) zurück."""
    content_raw = table.get('content') or '{}'
    try:
        content = json.loads(content_raw) if isinstance(content_raw, str) else content_raw
    except (json.JSONDecodeError, TypeError):
        return '', False
    headers   = content.get('headers') or content.get('columns') or []
    data_rows = content.get('data', [])
    has_data  = len(data_rows) > 0 and any(
        any(str(cell).strip() for cell in row) for row in data_rows
    )
    if not has_data:
        return ' '.join(str(h) for h in headers), False
    all_cells = [str(h) for h in headers]
    for row in data_rows:
        all_cells.extend(str(cell) for cell in row)
    return ' '.join(all_cells), True


# --- Zahlennormalisierung ---

def normalize_number(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r'[%±≈~<>≤≥°]', '', s)
    s = re.sub(r'(\d),(\d{3})', r'\1\2', s)   # 1,234 → 1234
    s = re.sub(r'(\d),(\d{1,2})$', r'\1.\2', s)  # 34,7 → 34.7
    return s.strip()


def extract_numbers_regex(text: str) -> set:
    if not text:
        return set()
    raw = re.findall(r'[\d]+(?:[.,]\d+)*(?:\s*%)?', text)
    numbers = set()
    for n in raw:
        n_clean = normalize_number(n)
        try:
            val = float(n_clean)
        except ValueError:
            continue
        if 2000 <= val <= 2030: continue       # Jahreszahl
        if val in (0.0, 1.0, 2.0) and val == int(val): continue  # zu generisch
        if val > 100_000: continue
        numbers.add(n_clean)
    return numbers


def extract_numbers_spacy(text: str) -> set:
    if not text or not SPACY_OK:
        return set()
    doc = _nlp(text[:20_000])
    numbers = set()
    for ent in doc.ents:
        if ent.label_ in ('CARDINAL', 'PERCENT', 'QUANTITY'):
            n = normalize_number(ent.text)
            try:
                val = float(n)
            except ValueError:
                continue
            if 2000 <= val <= 2030: continue
            if val in (0.0, 1.0, 2.0) and val == int(val): continue
            numbers.add(n)
    return numbers


def fuzzy_match(table_nums: set, text_nums: set, tolerance: float = 0.05) -> int:
    matched = 0
    for tn in table_nums:
        try: tv = float(tn)
        except ValueError: continue
        for xn in text_nums:
            try:
                xv = float(xn)
                if tv == 0:
                    if xv == 0: matched += 1; break
                elif abs(tv - xv) / abs(tv) <= tolerance:
                    matched += 1; break
            except ValueError:
                continue
    return matched


# --- Textfenster vorwärts ---

def get_window_forward(full_text: str, reference: str, size: int = 500) -> str:
    ref_pos = full_text.find(reference)
    if ref_pos >= 0:
        return full_text[ref_pos: ref_pos + size]
    short = reference[:60].strip()
    pos = full_text.find(short)
    if pos >= 0:
        return full_text[pos: pos + size]
    return reference


print('✓ Hilfsfunktionen geladen')

✓ Hilfsfunktionen geladen


## 4. Klassifikatoren (Stufe 1–3)

In [6]:
# --- Stufe 1: Regex ---

def classify_regex(table: dict, full_text: str) -> tuple:
    refs = table.get('references') or []
    if not refs:
        return 'no_refs', 'high'

    window_combined  = ' '.join(get_window_forward(full_text, r) for r in refs)
    all_refs_text    = ' '.join(refs)
    ref_word_count   = len(all_refs_text.split())
    table_text, is_complete = extract_table_content(table)
    table_nums  = extract_numbers_regex(table_text)
    window_nums = extract_numbers_regex(window_combined)

    overlap_score = (fuzzy_match(table_nums, window_nums) / len(table_nums)
                     if table_nums else 0.0)

    if ref_word_count < 15 and overlap_score < 0.1:
        return 'only_table', 'high'
    if not is_complete and ref_word_count < 25:
        return 'only_table', 'low'
    if overlap_score >= 0.30 and ref_word_count >= 15:
        return 'info_in_text', 'high'
    if overlap_score >= 0.15 and ref_word_count >= 40:
        return 'info_in_text', 'low'
    if 0.05 <= overlap_score < 0.30:
        return 'partial', 'low'
    if overlap_score < 0.05 and ref_word_count >= 15:
        return 'only_table', 'low'
    return 'partial', 'low'


# --- Stufe 2: spaCy ---

def classify_spacy(table: dict, full_text: str, regex_label: str) -> str:
    if not SPACY_OK:
        return regex_label
    refs = table.get('references') or []
    ref_word_count  = len(' '.join(refs).split())
    window_combined = ' '.join(get_window_forward(full_text, r) for r in refs)
    table_text, _   = extract_table_content(table)
    table_nums  = extract_numbers_spacy(table_text)
    window_nums = extract_numbers_spacy(window_combined)
    overlap = (fuzzy_match(table_nums, window_nums) / len(table_nums)
               if table_nums else 0.0)

    if overlap >= 0.25 and ref_word_count >= 15: return 'info_in_text'
    if overlap < 0.05  and ref_word_count <  20: return 'only_table'
    if ref_word_count >= 20 and overlap < 0.10:  return 'partial'
    return regex_label


# --- Stufe 3: Ollama-LLM ---

_LLM_PROMPT = """You are a scientific text analyst classifying how much a table's information appears in the surrounding text.

LABEL DEFINITIONS:
- info_in_text : Concrete numerical values from the table are explicitly repeated in the text
- partial      : The text references the table and describes it qualitatively, but rarely repeats exact numbers
- only_table   : The text only points to the table ("see Table 1") without describing its content
- no_refs      : No in-text reference to this table at all

TABLE CAPTION: {caption}
TABLE CONTENT (first rows):
{table_content}
IN-TEXT REFERENCES:
{refs}
TEXT WINDOW (500 chars after reference):
{window}

FEW-SHOT EXAMPLES:
  Table: Method|Score / Regex|0.33 / spaCy|0.41  |  Ref: "spaCy achieved 0.41 and Regex scored 0.33"  →  info_in_text
  Table: Gene|p-value / BRCA1|0.001             |  Ref: "Table 3 summarises differential expression"    →  partial
  Table: Sample|Conc / A|5.2mM                  |  Ref: "See Table 1."                                  →  only_table

Respond with EXACTLY one word: info_in_text / partial / only_table / no_refs
Label:"""

_VALID_LABELS = {'info_in_text', 'partial', 'only_table', 'no_refs'}

def classify_llm(table: dict, full_text: str, prev_label: str) -> str:
    if not OLLAMA_OK:
        return prev_label
    refs = table.get('references') or []
    caption = table.get('caption') or table.get('name') or ''
    table_text, _ = extract_table_content(table)
    windows = [get_window_forward(full_text, r) for r in refs]
    best_window = max(windows, key=len) if windows else ''
    try:
        content = json.loads(table.get('content') or '{}')
        headers = content.get('headers') or content.get('columns') or []
        rows    = content.get('data', [])[:6]
        table_repr = ' | '.join(str(h) for h in headers)
        table_repr += '\n' + '\n'.join(' | '.join(str(c) for c in r) for r in rows)
    except Exception:
        table_repr = table_text[:400]

    prompt = _LLM_PROMPT.format(
        caption=caption[:200],
        table_content=table_repr[:600],
        refs=(' '.join(refs))[:500],
        window=best_window[:500],
    )
    try:
        response = _ollama_lib.chat(
            model=_ollama_model,
            messages=[{'role': 'user', 'content': prompt}],
            options={'temperature': 0.0},
        )
        raw = response['message']['content'].strip().lower()
        for label in _VALID_LABELS:
            if label in raw:
                return label
        return prev_label
    except Exception:
        return prev_label


# --- Kaskade ---

def classify_table_cascade(table: dict, full_text: str) -> tuple:
    regex_label, confidence = classify_regex(table, full_text)
    if confidence == 'high':
        return regex_label, 'regex'
    after_spacy = classify_spacy(table, full_text, regex_label) if SPACY_OK else regex_label
    method = 'spacy' if SPACY_OK else 'regex_only'
    if OLLAMA_OK and after_spacy in ('partial', 'info_in_text'):
        return classify_llm(table, full_text, after_spacy), 'llm'
    return after_spacy, method


print('✓ Klassifikatoren geladen')

✓ Klassifikatoren geladen


## 5. Pipeline ausführen

In [7]:
rows = []
table_global_idx = 0

for fpath in tqdm(json_files, desc='Pipeline'):
    try:
        with open(fpath, encoding='utf-8') as f:
            doc = json.load(f)
    except Exception:
        continue

    full_text = doc.get('text', '')
    doi = doc.get('doi', fpath.stem)

    for i, table in enumerate(doc.get('tables', [])):
        label, method = classify_table_cascade(table, full_text)
        _, is_complete = extract_table_content(table)
        rows.append({
            'idx':             table_global_idx,
            'file':            fpath.name,
            'doi':             doi,
            'table_idx':       i,
            'predicted_label': label,
            'method_used':     method,
            'table_complete':  is_complete,
            'ref_count':       len(table.get('references') or []),
            'caption':         (table.get('caption') or '')[:80],
        })
        table_global_idx += 1

df = pd.DataFrame(rows)
df.to_csv(RESULTS_PATH, index=False)

print(f'\nTabellen gesamt : {len(df)}')
print(f'Gespeichert     : {RESULTS_PATH}')
print()
print('Label-Verteilung (Predictions):')
print(df['predicted_label'].value_counts().to_string())
print()
print('Methoden-Verteilung:')
print(df['method_used'].value_counts().to_string())

Pipeline:   0%|          | 0/499 [00:00<?, ?it/s]


Tabellen gesamt : 1121
Gespeichert     : /Users/cemilhantozak/Documents/Studium/6. Semester/DIS22 - Projekt ||/results_improved.csv

Label-Verteilung (Predictions):
predicted_label
no_refs         738
info_in_text    191
partial         126
only_table       66

Methoden-Verteilung:
method_used
regex    891
llm      201
spacy     29


## 6. Evaluation gegen manuelle Annotation (Cohen's Kappa)

Nur ausführbar wenn `annotation/alle_annotiert.csv` vorhanden ist (Spalten: `idx`, `manual`).

In [8]:
if not ANNOT_PATH.exists():
    print(f'Annotation-Datei nicht gefunden: {ANNOT_PATH}')      
    print('Bitte lege alle_annotiert.csv in den Ordner annotation/ und führe diese Zelle erneut aus.')
else:
    from sklearn.metrics import cohen_kappa_score, classification_report

    gt = pd.read_csv(ANNOT_PATH)
    merged = df.merge(gt[['idx', 'manual']], on='idx', how='inner')

    if len(merged) == 0:
        print('FEHLER: Keine Überschneidung zwischen Predictions und Ground Truth.')
        print(f'  Predictions idx: {df["idx"].min()}–{df["idx"].max()}')
        print(f'  Annotation  idx: {gt["idx"].min()}–{gt["idx"].max()}')
    else:
        y_true = merged['manual']
        y_pred = merged['predicted_label']
        kappa  = cohen_kappa_score(y_true, y_pred)

        print('=' * 55)
        print('EVALUATION ERGEBNISSE')
        print('=' * 55)
        print(f'Cohen\'s Kappa : κ = {kappa:.3f}  (Original Regex: κ = 0.33)')
        print(f'Accuracy      : {(y_true == y_pred).mean():.1%}  ({len(merged)} Tabellen)')
        print()

        print('Label-Verteilung (True vs Predicted):')
        dist = pd.DataFrame({
            'True':      y_true.value_counts(),
            'Predicted': y_pred.value_counts(),
        }).fillna(0).astype(int)
        print(dist.to_string())
        print()

        print('Classification Report:')
        print(classification_report(y_true, y_pred, zero_division=0))

        print('Methoden-Verteilung:')
        print(merged['method_used'].value_counts().to_string())

        print('\nTop-5 Verwechslungen:')
        conf = merged.groupby(['manual', 'predicted_label']).size().reset_index(name='n')
        wrong = conf[conf['manual'] != conf['predicted_label']].nlargest(5, 'n')
        print(wrong.to_string(index=False))

        interpretation = (
            'Schwach' if kappa < 0.2 else
            'Gering'  if kappa < 0.4 else
            'Moderat ✓' if kappa < 0.6 else
            'Gut ✓✓'   if kappa < 0.8 else
            'Sehr gut ✓✓✓'
        )
        print(f'\nFazit: κ = {kappa:.3f} → {interpretation}')

KeyError: "None of [Index(['idx', 'manual'], dtype='object')] are in the [columns]"